#### Assignment #1 - Part 1 - Average Salary of Teachers in Chicago
-   Data cleansing 
-   Select the right rows for teachers 
-   Delete na values or impute 
-   Winsorze for outliers and compute summary stats

In [2]:
import pandas as pd
import numpy as np
from scipy.stats import mstats
import matplotlib.pyplot as plt
import matplotlib.pyplot as plt
import numpy
import pandas
import random
import sys

import seaborn
from datetime import datetime

In [3]:
tt = pandas.read_csv(r'data\cars.csv')
tt

,Make,Model,Type,Origin,DriveTrain,MSRP,Invoice,EngineSize,Cylinders,Horsepower,MPG_City,MPG_Highway,Weight,Wheelbase,Length
0,Acura,MDX,SUV,Asia,AWD,36945,33337,3.5,6.0,265,17,23,4451,106,189
1,Acura,RSX Type S 2dr,Sedan,Asia,FWD,23820,21761,2.0,4.0,200,24,31,2778,101,172
2,Acura,TSX 4dr,Sedan,Asia,FWD,26990,24647,2.4,4.0,200,22,29,3230,105,183
3,Acura,TL 4dr,Sedan,Asia,FWD,33195,30299,3.2,6.0,270,20,28,3575,108,186
4,Acura,3.5 RL 4dr,Sedan,Asia,FWD,43755,39014,3.5,6.0,225,18,24,3880,115,197
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
423,Volvo,C70 LPT convertible 2dr,Sedan,Europe,FWD,40565,38203,2.4,5.0,197,21,28,3450,105,186
424,Volvo,C70 HPT convertible 2dr,Sedan,Europe,FWD,42565,40083,2.3,5.0,242,20,26,3450,105,186
425,Volvo,S80 T6 4dr,Sedan,Europe,FWD,45210,42573,2.9,6.0,268,19,26,3653,110,190
426,Volvo,V40,Wagon,Europe,FWD,26135,24641,1.9,4.0,170,22,29,2822,101,180


In [4]:
import statsmodels.formula.api as smf

cars = pd.read_csv(r"data\cars.csv")
cols = ["MSRP", "DriveTrain", "Origin", "EngineSize", "Horsepower", "Length", "Weight", "Wheelbase"]
model_data = cars[cols].dropna()

model = smf.ols(
    "MSRP ~ C(DriveTrain) + C(Origin) + EngineSize + Horsepower + Length + Weight + Wheelbase",
    data=model_data,
).fit()

print(f"R2: {model.rsquared:.6f}")
print(f"Rows used: {len(model_data)}")

R2: 0.775129
Rows used: 428


In [7]:
import itertools
import math

predictors = [
    "DriveTrain",
    "Origin",
    "EngineSize",
    "Horsepower",
    "Length",
    "Weight",
    "Wheelbase",
]

base_terms = {
    "DriveTrain": "C(DriveTrain)",
    "Origin": "C(Origin)",
    "EngineSize": "EngineSize",
    "Horsepower": "Horsepower",
    "Length": "Length",
    "Weight": "Weight",
    "Wheelbase": "Wheelbase",
}

r2_cache = {(): 0.0}

for k in range(1, len(predictors) + 1):
    for combo in itertools.combinations(predictors, k):
        key = tuple(sorted(combo))
        terms = [base_terms[name] for name in combo]
        formula = "MSRP ~ " + " + ".join(terms)
        fitted = smf.ols(formula, data=model_data).fit()
        r2_cache[key] = fitted.rsquared

p = len(predictors)
fact = math.factorial

shapley = {}
for pred in predictors:
    value = 0.0
    others = [x for x in predictors if x != pred]
    for k in range(0, p):
        for subset in itertools.combinations(others, k):
            subset = tuple(sorted(subset))
            with_pred = tuple(sorted(subset + (pred,)))
            weight = fact(k) * fact(p - k - 1) / fact(p)
            value += weight * (r2_cache[with_pred] - r2_cache[subset])
    shapley[pred] = value

shapley_total = sum(shapley.values())
standardized = {k: v / shapley_total for k, v in shapley.items()}

print("Shapley values (R2 contribution):")
for k in predictors:
    print(f"  {k}: {shapley[k]:.6f}")

print("\nStandardized Shapley values:")
for k in predictors:
    print(f"  {k}: {standardized[k]:.6f}")

print(f"\nStandardized Shapley value for Length: {standardized['Length']:.6f}")

Shapley values (R2 contribution):
  DriveTrain: 0.069461
  Origin: 0.135576
  EngineSize: 0.121660
  Horsepower: 0.343276
  Length: 0.015408
  Weight: 0.064450
  Wheelbase: 0.025299

Standardized Shapley values:
  DriveTrain: 0.089612
  Origin: 0.174907
  EngineSize: 0.156954
  Horsepower: 0.442863
  Length: 0.019877
  Weight: 0.083147
  Wheelbase: 0.032639

Standardized Shapley value for Length: 0.019877


In [10]:
# sort dictionary standardized by values
sorted_standardized = dict(sorted(standardized.items(), key=lambda item: item[1], reverse=True))
print(sorted_standardized)

{'Horsepower': np.float64(0.4428632448476256), 'Origin': np.float64(0.17490728512332743), 'EngineSize': np.float64(0.15695414152044104), 'DriveTrain': np.float64(0.08961215794103357), 'Weight': np.float64(0.08314716517781678), 'Wheelbase': np.float64(0.03263858762356092), 'Length': np.float64(0.019877417766194574)}
